# Analyzing descriptor 
Analyzing CoP variables, extracted from force platform and radar stabilograms

In [1]:
import pandas as pd
import pingouin as pg
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
df_platform_features = pd.read_csv("../results/platform_descriptors.csv", index_col=0)
df_radar_features = pd.read_csv("../results/radar_descriptors.csv", index_col=0)

## Compute variables error 
with platform variables as reference

In [4]:
variables     = df_platform_features.columns.tolist()
acquisitions  = df_platform_features.index.tolist()

In [5]:
with np.errstate(divide="ignore", invalid="ignore"):
    rel_error = np.where(
        df_platform_features.values == 0,
        np.nan,
        np.abs(df_radar_features.values - df_platform_features.values) / np.abs(df_platform_features.values) * 100
    )
 
error_df = pd.DataFrame(rel_error, index=acquisitions, columns=variables)

error_df

,mean_value_ML,mean_value_AP,mean_distance_ML,mean_distance_AP,mean_distance_Radius,maximal_distance_ML,maximal_distance_AP,maximal_distance_Radius,rms_ML,rms_AP,...,critical_time_Diffusion_ML,critical_displacement_Diffusion_ML,short_time_scaling_Diffusion_ML,long_time_scaling_Diffusion_ML,short_time_diffusion_Diffusion_AP,long_time_diffusion_Diffusion_AP,critical_time_Diffusion_AP,critical_displacement_Diffusion_AP,short_time_scaling_Diffusion_AP,long_time_scaling_Diffusion_AP
0,28.397062,27.575847,5.309202,4.677621,4.473761,20.177286,27.632249,35.550498,11.966703,8.359268,...,6.493205,3.271733e+01,1.214516,51.482456,37.406310,98.642997,5.369135,44.553959,0.959244,59.047704
1,288.244641,81.150750,9.357027,20.941178,5.737107,2.238097,26.417775,15.668590,6.802487,30.688649,...,0.134924,3.058712e+01,3.837973,34.486670,37.764582,90.638306,0.078775,38.467429,0.823593,66.828483
2,106.039168,107.822586,17.001450,8.239039,12.011459,17.798840,4.224432,0.401384,21.183726,9.052078,...,27.218361,1.079846e+01,2.217631,33.919724,48.729305,98.055458,11.086515,57.788263,2.794913,71.232543
3,71.221398,2680.677126,11.597603,25.830788,12.144392,12.807061,1.926632,3.686857,9.688086,23.444469,...,4.530295,3.172556e+01,0.707116,14.428311,45.576143,74.784220,17.773081,39.644257,15.285316,50.594005
4,7808.838475,1501.681555,13221.776319,270.757113,4576.662910,11782.218350,270.479920,5800.280480,12018.053372,276.394259,...,124.827627,2.852099e+06,11.146828,18.326461,529.184423,131.994048,40.524266,142.784532,1.539781,221.902928


In [10]:
summary = pd.DataFrame({
    "mean_error_%"   : error_df.mean(),
    "std_error_%"    : error_df.std(),
    "median_error_%" : error_df.median(),
    "max_error_%"    : error_df.max(),
    "n_below_20pct"  : (error_df < 20).sum(),
    "n_below_20pct_%" : (error_df < 20).mean() * 100,
})

summary


,mean_error_%,std_error_%,median_error_%,max_error_%,n_below_20pct,n_below_20pct_%
mean_value_ML,1660.548149,3438.424993,106.039168,7808.838475,0,0.0
mean_value_AP,879.781573,1182.155960,107.822586,2680.677126,0,0.0
mean_distance_ML,2653.008320,5908.122430,11.597603,13221.776319,4,80.0
mean_distance_AP,66.089148,114.745437,20.941178,270.757113,2,40.0
mean_distance_Radius,922.205926,2042.906583,12.011459,4576.662910,4,80.0
...,...,...,...,...,...,...
long_time_diffusion_Diffusion_AP,98.823006,20.891339,98.055458,131.994048,0,0.0
critical_time_Diffusion_AP,14.966354,15.731214,11.086515,40.524266,4,80.0
critical_displacement_Diffusion_AP,64.647688,44.347361,44.553959,142.784532,0,0.0
short_time_scaling_Diffusion_AP,4.280570,6.200916,1.539781,15.285316,5,100.0


## Compute ICC

In [ ]:
features = [col for col in df_platform_features.columns if col != 'type']

icc_results = {}

for feat in features:
    radar_vals = df_radar_features[feat].values
    force_vals = df_platform_features[feat].values
    
    # Créer DataFrame pour pingouin
    df_feat = pd.DataFrame({
        'Radar': radar_vals,
        'Force': force_vals
    })
    
    # Préparer pour pingouin
    df_tmp = df_feat.melt(var_name='Instrument', value_name='Score')
    df_tmp['Repetition'] = list(range(1, len(radar_vals)+1)) * 2  # répéter pour Radar et Force
    
    # Calculer ICC
    icc = pg.intraclass_corr(
        data=df_tmp,
        targets='Repetition',
        raters='Instrument',
        ratings='Score'
    )
    
    # Récupérer l'ICC de type "ICC2" (two-way, consistency, single) par exemple
    icc_value = icc.loc[icc['Type']=='ICC2','ICC'].values[0]
    icc_results[feat] = icc_value

# Convertir en DataFrame pour visualiser facilement
icc_df = pd.DataFrame.from_dict(icc_results, orient='index', columns=['ICC'])
icc_df = icc_df.sort_values('ICC', ascending=False)

print(icc_df)

                                                        ICC
energy_content_below_05_Power_Spectrum_Density_AP  0.865091
total_power_Power_Spectrum_Density_AP              0.859028
long_time_scaling_Diffusion_ML                     0.826198
mean_velocity_AP                                   0.825075
peak_velocity_all_SPD_AP                           0.794358
...                                                     ...
mean_distance_Radius                              -0.294352
mean_velocity_ML_AND_AP                           -0.429000
principal_sway_direction_ML_AND_AP                -0.595309
mean_distance_peak_Sway_Density                   -0.674343
sway_area_per_second_ML_AND_AP                    -0.745834

[72 rows x 1 columns]


In [24]:
icc_df
icc_df.to_csv("../results/icc.csv")